# Week 6 - Spark Architecture & Transformations

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Week 6 Assignment") \
    .getOrCreate()

print("Spark Started Successfully!")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/29 18:11:23 WARN Utils: Your hostname, Himanshus-MacBook-Air-2.local, resolves to a loopback address: 127.0.0.1; using 10.125.113.160 instead (on interface en0)
26/06/29 18:11:23 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/29 18:11:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark Started Successfully!


## Creating Sample Dataset

In [2]:

data = [
    (101, "Laptop", "Electronics", 50000, "North", "Completed", 1200),
    (102, "Mouse", "Electronics", 1500, "South", "Pending", 500),
    (103, "Chair", "Furniture", 7000, "North", "Completed", 2200),
    (104, "Table", "Furniture", 12000, "West", "Completed", 900),
    (105, "Phone", "Electronics", 30000, "East", "Completed", 4500),
    (106, "Keyboard", "Electronics", 2500, "North", "Pending", 700),
    (107, "TV", "Electronics", 55000, "West", "Completed", 5000),
    (108, "Sofa", "Furniture", 25000, "South", "Completed", 1800)
]

columns = [
    "product_id",
    "product_name",
    "category",
    "price",
    "region",
    "status",
    "amount"
]

df = spark.createDataFrame(data, columns)

df.show()

+----------+------------+-----------+-----+------+---------+------+
|product_id|product_name|   category|price|region|   status|amount|
+----------+------------+-----------+-----+------+---------+------+
|       101|      Laptop|Electronics|50000| North|Completed|  1200|
|       102|       Mouse|Electronics| 1500| South|  Pending|   500|
|       103|       Chair|  Furniture| 7000| North|Completed|  2200|
|       104|       Table|  Furniture|12000|  West|Completed|   900|
|       105|       Phone|Electronics|30000|  East|Completed|  4500|
|       106|    Keyboard|Electronics| 2500| North|  Pending|   700|
|       107|          TV|Electronics|55000|  West|Completed|  5000|
|       108|        Sofa|  Furniture|25000| South|Completed|  1800|
+----------+------------+-----------+-----+------+---------+------+



## Schema is -

In [3]:
df.printSchema()

root
 |-- product_id: long (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: long (nullable = true)
 |-- region: string (nullable = true)
 |-- status: string (nullable = true)
 |-- amount: long (nullable = true)



In [4]:
df.show(5)

+----------+------------+-----------+-----+------+---------+------+
|product_id|product_name|   category|price|region|   status|amount|
+----------+------------+-----------+-----+------+---------+------+
|       101|      Laptop|Electronics|50000| North|Completed|  1200|
|       102|       Mouse|Electronics| 1500| South|  Pending|   500|
|       103|       Chair|  Furniture| 7000| North|Completed|  2200|
|       104|       Table|  Furniture|12000|  West|Completed|   900|
|       105|       Phone|Electronics|30000|  East|Completed|  4500|
+----------+------------+-----------+-----+------+---------+------+
only showing top 5 rows


## creating dataset.csv

In [5]:
df.toPandas().to_csv("dataset.csv", index=False)

# Q1. Explain the roles of Driver, Cluster Manager, and Executor in a Spark application.

**Answer:**

- **Driver:** The Driver is the brain of a Spark application. It creates the SparkSession, builds the execution plan (DAG), schedules tasks, and collects results from executors.

- **Cluster Manager:** The Cluster Manager allocates resources (CPU and memory) and launches executors on worker nodes.

- **Executor:** Executors run the tasks assigned by the Driver, process partitions of data, and return the results to the Driver.

# Q2. How does Spark's Lazy Evaluation improve performance?

**Answer:**

Spark does not execute transformations immediately. Instead, it records them in a DAG (Directed Acyclic Graph). Execution starts only when an action such as `show()`, `count()`, or `collect()` is called. This allows Spark to optimize the execution plan, reduce unnecessary work, and improve performance.

In [6]:
filtered_df = df.filter(df.price > 5000)

selected_df = filtered_df.select("product_name", "price")

selected_df.show()

+------------+-----+
|product_name|price|
+------------+-----+
|      Laptop|50000|
|       Chair| 7000|
|       Table|12000|
|       Phone|30000|
|          TV|55000|
|        Sofa|25000|
+------------+-----+



# Q3. Read a CSV file with header and inferSchema enabled.

In [7]:
csv_df = spark.read.csv(
    "dataset.csv",
    header=True,
    inferSchema=True
)

csv_df.show()

+----------+------------+-----------+-----+------+---------+------+
|product_id|product_name|   category|price|region|   status|amount|
+----------+------------+-----------+-----+------+---------+------+
|       101|      Laptop|Electronics|50000| North|Completed|  1200|
|       102|       Mouse|Electronics| 1500| South|  Pending|   500|
|       103|       Chair|  Furniture| 7000| North|Completed|  2200|
|       104|       Table|  Furniture|12000|  West|Completed|   900|
|       105|       Phone|Electronics|30000|  East|Completed|  4500|
|       106|    Keyboard|Electronics| 2500| North|  Pending|   700|
|       107|          TV|Electronics|55000|  West|Completed|  5000|
|       108|        Sofa|  Furniture|25000| South|Completed|  1800|
+----------+------------+-----------+-----+------+---------+------+



# Q4. Difference between CSV and Parquet

| CSV | Parquet |
|------|----------|
| Row-based storage | Column-based storage |
| Larger file size | Smaller due to compression |
| Slower to read | Faster to read |
| No schema information | Stores schema |
| Suitable for data exchange | Suitable for analytics and Spark processing |

# Q5. Select product_id and price where category is Electronics.

In [8]:
df.filter(df.category == "Electronics") \
  .select("product_id", "price") \
  .show()

+----------+-----+
|product_id|price|
+----------+-----+
|       101|50000|
|       102| 1500|
|       105|30000|
|       106| 2500|
|       107|55000|
+----------+-----+



# Q6. Rename the column old_name to new_name and cast the price column from String to Double.

In [9]:
from pyspark.sql.functions import col

df = df.withColumnRenamed("product_name", "new_name")

df = df.withColumn(
    "price",
    col("price").cast("double")
)

df.printSchema()
df.show()

root
 |-- product_id: long (nullable = true)
 |-- new_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- region: string (nullable = true)
 |-- status: string (nullable = true)
 |-- amount: long (nullable = true)

+----------+--------+-----------+-------+------+---------+------+
|product_id|new_name|   category|  price|region|   status|amount|
+----------+--------+-----------+-------+------+---------+------+
|       101|  Laptop|Electronics|50000.0| North|Completed|  1200|
|       102|   Mouse|Electronics| 1500.0| South|  Pending|   500|
|       103|   Chair|  Furniture| 7000.0| North|Completed|  2200|
|       104|   Table|  Furniture|12000.0|  West|Completed|   900|
|       105|   Phone|Electronics|30000.0|  East|Completed|  4500|
|       106|Keyboard|Electronics| 2500.0| North|  Pending|   700|
|       107|      TV|Electronics|55000.0|  West|Completed|  5000|
|       108|    Sofa|  Furniture|25000.0| South|Completed|  1800|

## here the new name has been converted to string

# Q7. How does Spark use the Lineage Graph (DAG) to provide fault tolerance?

**Answer:**

Spark records all transformations in a Directed Acyclic Graph (DAG). Instead of storing multiple copies of data, Spark remembers how the data was created. If a worker node fails and loses a partition, Spark recomputes only the missing partition using the DAG. This provides fault tolerance without storing duplicate data.

# Q8. Filter rows where status is 'Completed' and amount is greater than 1000.

In [10]:
df.filter(
    (df.status == "Completed") &
    (df.amount > 1000)
).show()

+----------+--------+-----------+-------+------+---------+------+
|product_id|new_name|   category|  price|region|   status|amount|
+----------+--------+-----------+-------+------+---------+------+
|       101|  Laptop|Electronics|50000.0| North|Completed|  1200|
|       103|   Chair|  Furniture| 7000.0| North|Completed|  2200|
|       105|   Phone|Electronics|30000.0|  East|Completed|  4500|
|       107|      TV|Electronics|55000.0|  West|Completed|  5000|
|       108|    Sofa|  Furniture|25000.0| South|Completed|  1800|
+----------+--------+-----------+-------+------+---------+------+



# Q9. Explain Predicate Pushdown.

**Answer:**

Predicate Pushdown is an optimization in Spark where filter conditions are pushed to the storage layer. Instead of reading the entire file, Spark reads only the required rows. This reduces disk I/O, memory usage, and improves performance. It works efficiently with columnar formats like Parquet.

# Q10. Add a new column final_price by adding 18% tax.

In [11]:
from pyspark.sql.functions import col

df = df.withColumn(
    "final_price",
    col("price") * 1.18
)

df.show()

+----------+--------+-----------+-------+------+---------+------+-----------+
|product_id|new_name|   category|  price|region|   status|amount|final_price|
+----------+--------+-----------+-------+------+---------+------+-----------+
|       101|  Laptop|Electronics|50000.0| North|Completed|  1200|    59000.0|
|       102|   Mouse|Electronics| 1500.0| South|  Pending|   500|     1770.0|
|       103|   Chair|  Furniture| 7000.0| North|Completed|  2200|     8260.0|
|       104|   Table|  Furniture|12000.0|  West|Completed|   900|    14160.0|
|       105|   Phone|Electronics|30000.0|  East|Completed|  4500|    35400.0|
|       106|Keyboard|Electronics| 2500.0| North|  Pending|   700|     2950.0|
|       107|      TV|Electronics|55000.0|  West|Completed|  5000|    64900.0|
|       108|    Sofa|  Furniture|25000.0| South|Completed|  1800|    29500.0|
+----------+--------+-----------+-------+------+---------+------+-----------+



# Q11. Difference between Transformations and Actions

**Transformation**
- Returns a new DataFrame.
- Executes lazily.
- Examples: filter(), select(), withColumn()

**Action**
- Triggers execution.
- Returns results or writes data.
- Examples: show(), count(), collect()

In [12]:
# Transformation
filtered_df = df.filter(df.price > 5000)

# Action
filtered_df.show()

+----------+--------+-----------+-------+------+---------+------+-----------+
|product_id|new_name|   category|  price|region|   status|amount|final_price|
+----------+--------+-----------+-------+------+---------+------+-----------+
|       101|  Laptop|Electronics|50000.0| North|Completed|  1200|    59000.0|
|       103|   Chair|  Furniture| 7000.0| North|Completed|  2200|     8260.0|
|       104|   Table|  Furniture|12000.0|  West|Completed|   900|    14160.0|
|       105|   Phone|Electronics|30000.0|  East|Completed|  4500|    35400.0|
|       107|      TV|Electronics|55000.0|  West|Completed|  5000|    64900.0|
|       108|    Sofa|  Furniture|25000.0| South|Completed|  1800|    29500.0|
+----------+--------+-----------+-------+------+---------+------+-----------+



# Q12. Read a Parquet file, remove rows with null user_id, and save as CSV.

## first we will creat a parquet file

In [13]:
df.write.mode("overwrite").parquet("sample_parquet")

26/06/29 18:23:36 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


## now reading the parquet file

In [14]:
parquet_df = spark.read.parquet("sample_parquet")

parquet_df.show()

+----------+--------+-----------+-------+------+---------+------+-----------+
|product_id|new_name|   category|  price|region|   status|amount|final_price|
+----------+--------+-----------+-------+------+---------+------+-----------+
|       101|  Laptop|Electronics|50000.0| North|Completed|  1200|    59000.0|
|       106|Keyboard|Electronics| 2500.0| North|  Pending|   700|     2950.0|
|       105|   Phone|Electronics|30000.0|  East|Completed|  4500|    35400.0|
|       102|   Mouse|Electronics| 1500.0| South|  Pending|   500|     1770.0|
|       103|   Chair|  Furniture| 7000.0| North|Completed|  2200|     8260.0|
|       108|    Sofa|  Furniture|25000.0| South|Completed|  1800|    29500.0|
|       104|   Table|  Furniture|12000.0|  West|Completed|   900|    14160.0|
|       107|      TV|Electronics|55000.0|  West|Completed|  5000|    64900.0|
+----------+--------+-----------+-------+------+---------+------+-----------+



## now removing the rows with null user id -> filtering

In [15]:
filtered_df = parquet_df.filter(
    parquet_df.product_id.isNotNull()
)

filtered_df.show()

+----------+--------+-----------+-------+------+---------+------+-----------+
|product_id|new_name|   category|  price|region|   status|amount|final_price|
+----------+--------+-----------+-------+------+---------+------+-----------+
|       101|  Laptop|Electronics|50000.0| North|Completed|  1200|    59000.0|
|       106|Keyboard|Electronics| 2500.0| North|  Pending|   700|     2950.0|
|       105|   Phone|Electronics|30000.0|  East|Completed|  4500|    35400.0|
|       102|   Mouse|Electronics| 1500.0| South|  Pending|   500|     1770.0|
|       103|   Chair|  Furniture| 7000.0| North|Completed|  2200|     8260.0|
|       108|    Sofa|  Furniture|25000.0| South|Completed|  1800|    29500.0|
|       104|   Table|  Furniture|12000.0|  West|Completed|   900|    14160.0|
|       107|      TV|Electronics|55000.0|  West|Completed|  5000|    64900.0|
+----------+--------+-----------+-------+------+---------+------+-----------+



## now again change it to .csv file

In [16]:
filtered_df.write.mode("overwrite").csv(
    "output_csv",
    header=True
)

# Q13. Difference between Client Mode and Cluster Mode

| Client Mode | Cluster Mode |
|--------------|--------------|
| Driver runs on local machine | Driver runs inside cluster |
| Easy for development | Best for production |
| If client stops, job stops | Job continues even if client disconnects |

# Q14. Filter rows where region is North OR status is Completed.

In [17]:
df.filter(
    (df.region == "North") |
    (df.status == "Completed")
).show()

+----------+--------+-----------+-------+------+---------+------+-----------+
|product_id|new_name|   category|  price|region|   status|amount|final_price|
+----------+--------+-----------+-------+------+---------+------+-----------+
|       101|  Laptop|Electronics|50000.0| North|Completed|  1200|    59000.0|
|       103|   Chair|  Furniture| 7000.0| North|Completed|  2200|     8260.0|
|       104|   Table|  Furniture|12000.0|  West|Completed|   900|    14160.0|
|       105|   Phone|Electronics|30000.0|  East|Completed|  4500|    35400.0|
|       106|Keyboard|Electronics| 2500.0| North|  Pending|   700|     2950.0|
|       107|      TV|Electronics|55000.0|  West|Completed|  5000|    64900.0|
|       108|    Sofa|  Furniture|25000.0| South|Completed|  1800|    29500.0|
+----------+--------+-----------+-------+------+---------+------+-----------+



# Q15. Why use show(5) instead of collect()?

**Answer:**

show(5) displays only a few rows and is memory efficient.

collect() brings the entire dataset to the Driver node. For large datasets, this can consume huge memory and even crash the application.

Therefore, show() is preferred while exploring large datasets.

# Final Spark Data Pipeline

In [18]:
pipeline = spark.read.csv(
    "dataset.csv",
    header=True,
    inferSchema=True
)

pipeline = pipeline.withColumn(
    "price",
    col("price").cast("double")
)

pipeline = pipeline.filter(
    pipeline.price > 5000
)

pipeline = pipeline.withColumn(
    "GST",
    col("price") * 0.18
)

pipeline.show()

+----------+------------+-----------+-------+------+---------+------+------+
|product_id|product_name|   category|  price|region|   status|amount|   GST|
+----------+------------+-----------+-------+------+---------+------+------+
|       101|      Laptop|Electronics|50000.0| North|Completed|  1200|9000.0|
|       103|       Chair|  Furniture| 7000.0| North|Completed|  2200|1260.0|
|       104|       Table|  Furniture|12000.0|  West|Completed|   900|2160.0|
|       105|       Phone|Electronics|30000.0|  East|Completed|  4500|5400.0|
|       107|          TV|Electronics|55000.0|  West|Completed|  5000|9900.0|
|       108|        Sofa|  Furniture|25000.0| South|Completed|  1800|4500.0|
+----------+------------+-----------+-------+------+---------+------+------+



## saving as csv

In [19]:
pipeline.write.mode("overwrite").csv(
    "final_output_csv",
    header=True
)

## saving as parquet

In [20]:
pipeline.write.mode("overwrite").parquet(
    "final_output_parquet"
)

# Observations

- Spark architecture consists of Driver, Cluster Manager, and Executors.
- Spark uses Lazy Evaluation to optimize execution.
- CSV is row-based while Parquet is column-based and more efficient.
- Transformations are lazy, whereas Actions trigger execution.
- Predicate Pushdown improves query performance by reading only required data.
- Spark uses DAG (Lineage Graph) for fault tolerance.
- A complete Spark pipeline was implemented:
  Read → Transform → Filter → Write.
- Processed data was successfully saved in both CSV and Parquet formats.